# RL Lab 1

#### Required libraries

In [2]:
# If needed in a fresh environment:

import numpy as np
import matplotlib.pyplot as plt
import gymnasium as gym


## Concept recap
Agent interacts with env; at each step: state → action → reward → next state; an episode ends at terminal. Return is cumulative reward; γ decides how much we care about future.

### 1. Warm-up: Explore FrozenLake (states/actions/episode)

In [3]:
env = gym.make("FrozenLake-v1", is_slippery=True, render_mode="ansi")  # stochastic transitions
print("Observation space:", env.observation_space)
print("Action space:", env.action_space)



obs, info = env.reset()
print("Initial state:", obs, info)
print(env.render())


Observation space: Discrete(16)
Action space: Discrete(4)
Initial state: 0 {'prob': 1}

SFFF
FHFH
FFFH
HFFG



Run 1 random episode and print the trajectory (s, a, r, s’).

In [4]:
def run_random_episode(env, max_steps=100):
    obs, info = env.reset()
    traj = []
    total_reward = 0.0
    for t in range(max_steps):
        a = env.action_space.sample()
        next_obs, r, terminated, truncated, info = env.step(a)
        traj.append((obs, a, r, next_obs))
        total_reward += r
        obs = next_obs
        if terminated or truncated:
            break
    return traj, total_reward

traj, G = run_random_episode(env)
print("Episode length:", len(traj), "Return:", G)
print("First 10 transitions:", traj[:10])


#print the environment after the agent has done the episode
print(env.render())

# just printed some clarifications
print(f"\nS = Start (state 0)\nF = Frozen (safe)\nH = Hole (terminal, reward 0)\nG = Goal (state 15, reward 1)")


Episode length: 5 Return: 0.0
First 10 transitions: [(0, np.int64(0), 0, 4), (4, np.int64(0), 0, 8), (8, np.int64(2), 0, 9), (9, np.int64(1), 0, 13), (13, np.int64(1), 0, 12)]
  (Down)
SFFF
FHFH
FFFH
HFFG


S = Start (state 0)
F = Frozen (safe)
H = Hole (terminal, reward 0)
G = Goal (state 15, reward 1)


#### Questions:

What is the state in FrozenLake? What is an action?

When does an episode end in this environment?

Why might the same action from the same state lead to different outcomes (stochasticity / transition probability)?

## Answer:

**What is the state in FrozenLake?**

The state is the agents position on a grid (the frozen lake), the start sate is 0.

**What is an action?**

An action is what the agent can "do", it can move to another state using actions, in this case im guessing its 4 integers 0=LEFT, 1=DOWN, 2=RIGHT, 3=UP. as the `env.action_space` says its: Discrete(4). And we have to remember that the action is only an attempt att going to a new state and can have different probabilities.

**When does an episode end?**

An episode ends when the agent reaches the goal or a hole or the max steps(if set) in this case we see that the agent has fell in to a hole 

**Why might the same action lead to different outcomes?**

Because `is_slippery=True` it gets trickier for the agent to move as it wants and so the chance of the intended action to happen is 1/3 (33.3%). And the probability that the agent moves 2/3 (66.6%) the agent perpendicular to the intended direction is also 1/3 (33.3%) each way. So the outcome in this case depends on the fact that the ice is slippery but in some other case where the environment might be different the probabilities of different outcomes might change. 

### 2. Return and discount factor (γ) with a simple reward list

In [5]:
def discounted_return(rewards, gamma):
    G = 0.0
    for t, r in enumerate(rewards):
        G += (gamma**t) * r
    return G

rewards = [1, 1, -1, 1]
for gamma in [0.2, 0.9, 1.0]:
    print(gamma, discounted_return(rewards, gamma))


0.2 1.168
0.9 1.819
1.0 2.0


#### Exercise:

Create two reward sequences with the same sum but different timing (early vs late rewards).

Compare discounted returns for γ=0.2 vs γ=0.9.

#### Questions:

In your own words, what changes when γ is small vs large?

In [6]:
# Exercise

# Two sequences with the same sum (total = 10) but different timing
early_rewards = [5, 3, 2, 0, 0]   # big reward comes early
late_rewards  = [0, 0, 2, 3, 5]   # big reward comes late

for gamma in [0.2, 0.9]:
    G_early = discounted_return(early_rewards, gamma)
    G_late  = discounted_return(late_rewards,  gamma)
    print(f"γ={gamma}  |  early G={G_early:.4f}  |  late G={G_late:.4f}\n")



γ=0.2  |  early G=5.6800  |  late G=0.1120

γ=0.9  |  early G=9.3200  |  late G=7.0875



## Answer

When γ is small (for example, γ = 0.2), the agent focuses mostly on immediate rewards.  
This means rewards that happen later are heavily discounted (reduced), so early rewards contribute much more to the return.

So with a low γ:
- the early-reward sequence gets a higher discounted return,
- the late-reward sequence gets a much lower discounted return.

When γ is large (for example, γ = 0.9), future rewards are still important in some degree. The discounting is slower, so the difference between early and late reward timing becomes smaller.  
Early rewards are still slightly better, but late rewards are no longer ignored.

### 3. Tiny GridWorld (A, B, C terminal)

This implements a minimal deterministic grid:

- States: A (top-left), B (top-right), C (bottom-right, terminal), X blocked (bottom-left)

- Actions: 0=LEFT, 1=DOWN, 2=RIGHT, 3=UP

- Reward: normal move = -1, moving into C = +10

- γ will be used later

In [7]:
from dataclasses import dataclass

LEFT, DOWN, RIGHT, UP = 0, 1, 2, 3
ACTIONS = [LEFT, DOWN, RIGHT, UP]
ACTION_NAMES = {LEFT:"L", DOWN:"D", RIGHT:"R", UP:"U"}

@dataclass(frozen=True)
class TinyGrid:
    # 2x2 with one blocked cell:
    # (0,0)=A, (0,1)=B
    # (1,0)=X blocked, (1,1)=C terminal
    terminal = (1,1)
    blocked = (1,0)
    start_positions = {(0,0):"A", (0,1):"B", (1,1):"C"}

    def states(self):
        return [(0,0),(0,1),(1,1)]  # exclude blocked

    def is_terminal(self, s):
        return s == self.terminal

    def step(self, s, a):
        if self.is_terminal(s):
            return s, 0.0, True

        r, c = s
        nr, nc = r, c
        if a == LEFT:  nc -= 1
        if a == RIGHT: nc += 1
        if a == UP:    nr -= 1
        if a == DOWN:  nr += 1

        # boundaries
        if nr < 0 or nr > 1 or nc < 0 or nc > 1:
            nr, nc = r, c

        # blocked
        if (nr, nc) == self.blocked:
            nr, nc = r, c

        s2 = (nr, nc)

        # rewards
        if s2 == self.terminal:
            reward = 10.0
            done = True
        else:
            reward = -1.0
            done = False

        return s2, reward, done

grid = TinyGrid()
print("States:", grid.states())
print("From A, DOWN ->", grid.step((0,0), DOWN))
print("From B, DOWN ->", grid.step((0,1), DOWN))  # into C




States: [(0, 0), (0, 1), (1, 1)]
From A, DOWN -> ((0, 0), -1.0, False)
From B, DOWN -> ((1, 1), 10.0, True)


#### Exercise: 

Simulate one episode from A using a fixed policy (e.g., always DOWN if possible else RIGHT). Print trajectory and return.

V(s): How good is this state if I follow policy π from here?

Q(s,a): How good is taking action a in state s, then following π?

In [8]:
s = (0, 0)
traj = []

def fixed_policy(grid, s): # Try down if it does not move us down we try right
    next_s, rew, done = grid.step(s, DOWN)
    if next_s != s:
        return DOWN
    return RIGHT

while not grid.is_terminal(s):
    action = fixed_policy(grid, s)
    next_s, r, d = grid.step(s, action)
    traj.append((s, action, r, next_s))
    s = next_s

for s, a, r, s2 in traj:
    print(f"state:{s}  |  action:{ACTION_NAMES[a]}  |  state':{s2}  |  reward:{r}")

rewards = [r for _, _, r, _ in traj] # get all the rewards in one array

γ = 0.7

print(f"\nUndiscounted return (γ=1.0): {sum(rewards)}")
print(f"Discounted return  (γ={γ}): {discounted_return(rewards, γ):.4f}")


state:(0, 0)  |  action:R  |  state':(0, 1)  |  reward:-1.0
state:(0, 1)  |  action:D  |  state':(1, 1)  |  reward:10.0

Undiscounted return (γ=1.0): 9.0
Discounted return  (γ=0.7): 6.0000


## Answer:

I choose γ to be 0.7 just for testing out 

### V(s) means: how good is a state?

V(s) is the total discounted return I expect collecting, starting from state s, following the fixed policy all the way to the end:
- V(A) = -1 + 0.7×10 = 6.0 (one -1 step to B, then +10 goal from B)
- V(B) = 10.0 (one step straight to goal)
- V(C) = 0.0 (already terminal)
- V(X) = was not a starting position and is blocked

### Q(s, a) means: how good is a state + action pair?

Q(s, a) is the return if I take a specific action `a` in state `s` first, then follow the policy for the rest:

#### From A, LEFT hits boundary (stays at A), UP hits boundary (stays at A), DOWN hits X blocked (stays at A), RIGHT -> B:

- Q(A, LEFT) = -1 + 0.7×6.0 = 3.2
- Q(A, RIGHT) = -1 + 0.7×10.0 = 6.0 <- best
- Q(A, UP) = -1 + 0.7×6.0 = 3.2
- Q(A, DOWN) = -1 + 0.7×6.0 = 3.2 (blocked, stays at A)

#### From B — RIGHT hits boundary (stays at B), UP hits boundary (stays at B), LEFT -> A, DOWN -> C:

- Q(B, LEFT) = -1 + 0.7×6.0 = 3.2
- Q(B, RIGHT) = -1 + 0.7×10.0 = 6.0
- Q(B, UP) = -1 + 0.7×10.0 = 6.0
- Q(B, DOWN) = 10.0  best

The environment in this exercise is very small which means there are very short episodes and and thats why there is not much difference between all these episodes. The big differences is the Q values where if we hit a boundary that makes us stay in the same position, we see a drop in the returns. 
